# ⚾ MLB Lineup Generator & Sequential LSTM Neural Network Optimizer
### End-to-End Play-by-Play Sequence Modeling, Lineup Optimization, and Monte Carlo Statistical Validation

This notebook provides a complete environment for:
1. **MLB Data Ingestion**: Fetching real 2024 player stats via `pybaseball`.
2. **Play-by-Play (PBP) Game Engine**: Realistic 9-inning game simulator with RE24 run expectancy, base-runner advancement, forced walks, double plays, sac flies, and situation tracking.
3. **Sequential Neural Network (LSTM / RNN)**: Modeling situation context (outs, base states) and previous batter outcome dependencies to predict expected run value added (`Run_Value_RE24`).
4. **LSTM Lineup Optimization**: Screening 9-batter sequence permutations through the trained LSTM sequence model.
5. **Benchmark Validation**: 10,000-game Monte Carlo simulation with paired t-tests benchmarked against the **2024 Yankees Starting Lineup** (`Gleyber Torres`, `Juan Soto`, `Aaron Judge`, `Austin Wells`, `Giancarlo Stanton`, `Jazz Chisholm Jr.`, `Anthony Rizzo`, `Anthony Volpe`, `Alex Verdugo`).

In [ ]:
# Step 1: Install Dependencies
!pip install -q scipy scikit-learn matplotlib pandas numpy tensorflow

In [ ]:
# Step 1b: Install PyBaseball from Custom Fork
!pip install --upgrade --force-reinstall git+https://github.com/MirajShah12/pybaseball.git@master

In [ ]:
# Step 2: Core Class Definitions (ConfigManager, PlayerModel, LineupModel, GameGenerator, LineupAnalysis)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from typing import List, Dict, Tuple, Any, Optional
from itertools import permutations
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

BENCHMARK_YANKEES_2024 = [
    'Gleyber Torres', 'Juan Soto', 'Aaron Judge', 'Austin Wells',
    'Giancarlo Stanton', 'Jazz Chisholm Jr.', 'Anthony Rizzo',
    'Anthony Volpe', 'Alex Verdugo'
]

class ConfigManager:
    def __init__(self):
        self.default_simulations = 1000
        self.default_innings = 9
        self.min_plate_appearances = 150
        self.base_matrix = [
            [0.48, 0.25, 0.10],  # Bases empty
            [0.87, 0.48, 0.21],  # Runner on 1st
            [1.12, 0.67, 0.31],  # Runner on 2nd
            [1.38, 0.86, 0.32],  # Runners on 1st & 2nd
            [1.55, 0.96, 0.42],  # Runner on 3rd
            [1.78, 1.31, 0.48],  # Runners on 1st & 3rd
            [2.04, 1.41, 0.67],  # Runners on 2nd & 3rd
            [2.69, 1.61, 0.96]   # Bases loaded
        ]
        self.base_states = [
            (0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0),
            (0, 0, 1), (1, 0, 1), (0, 1, 1), (1, 1, 1)
        ]

class PlayerModel:
    def __init__(self, player_data: pd.Series, config_manager=None):
        self.name = str(player_data.get('Name', 'Unknown'))
        self.team = str(player_data.get('Team', player_data.get('Tm', 'Unknown')))
        self.pa = max(int(player_data.get('PA', 1)), 1)
        
        self.singles = float(player_data.get('1B', 0))
        self.doubles = float(player_data.get('2B', 0))
        self.triples = float(player_data.get('3B', 0))
        self.home_runs = float(player_data.get('HR', 0))
        self.walks = float(player_data.get('BB', 0))
        self.hit_by_pitch = float(player_data.get('HBP', 0))
        self.strikeouts = float(player_data.get('SO', 0))
        self.ground_into_double_play = float(player_data.get('GDP', 0))
        
        self.woba = float(player_data.get('wOBA', 0.320))
        self.obp = float(player_data.get('OBP', 0.320))
        self.slg = float(player_data.get('SLG', 0.400))
        self.xwoba = float(player_data.get('xwOBA', self.woba))
        self.xba = float(player_data.get('xBA', 0.250))
        self.xslg = float(player_data.get('xSLG', self.slg))
        self.iso = float(player_data.get('ISO', self.slg - self.obp + 0.05))
        
        self._calculate_derived_stats()
        
    def _calculate_derived_stats(self):
        self.single_rate = self.singles / self.pa
        self.double_rate = self.doubles / self.pa
        self.triple_rate = self.triples / self.pa
        self.hr_rate = self.home_runs / self.pa
        self.walk_rate = self.walks / self.pa
        self.hbp_rate = self.hit_by_pitch / self.pa
        self.strikeout_rate = self.strikeouts / self.pa
        self.gidp_rate = self.ground_into_double_play / self.pa
        self.contact_rate = max(0.0, 1.0 - self.strikeout_rate)
        self.power_score = self.slg - self.obp
        
    def get_outcome_probabilities(self) -> Dict[str, float]:
        sac_fly_prob = min(0.02, max(0.005, self.contact_rate * 0.02))
        groundout_prob = 0.24
        flyout_prob = 0.16
        
        total_positive = (self.single_rate + self.double_rate + self.triple_rate + self.hr_rate +
                         self.walk_rate + self.hbp_rate + self.strikeout_rate + sac_fly_prob +
                         groundout_prob + flyout_prob)
        remaining_out_prob = max(0.0, 1.0 - total_positive)
        
        probs = {
            'single': self.single_rate, 'double': self.double_rate, 'triple': self.triple_rate, 'hr': self.hr_rate,
            'walk': self.walk_rate, 'hbp': self.hbp_rate, 'strikeout': self.strikeout_rate,
            'sac_fly': sac_fly_prob, 'groundout': groundout_prob, 'flyout': flyout_prob, 'out': remaining_out_prob
        }
        total = sum(probs.values())
        return {k: v / total for k, v in probs.items()} if total > 0 else probs
        
    def simulate_at_bat(self) -> str:
        probs = self.get_outcome_probabilities()
        return np.random.choice(list(probs.keys()), p=list(probs.values()))

class LineupModel:
    def __init__(self, players: List[PlayerModel], config_manager=None):
        self.players = players[:9]
        self.config_manager = config_manager
        self.lineup_order = [p.name for p in self.players]
        
    def get_player_by_name(self, name: str) -> Optional[PlayerModel]:
        for p in self.players:
            if p.name == name:
                return p
        return None
        
    def set_lineup_order(self, new_order: List[str]):
        self.lineup_order = new_order
        
    def _get_benchmark_order(self) -> List[str]:
        # Match 2024 Yankees starting lineup order if available
        bench_names = [n for n in BENCHMARK_YANKEES_2024 if self.get_player_by_name(n) is not None]
        rest_names = [p.name for p in self.players if p.name not in bench_names]
        return (bench_names + rest_names)[:9]

class GameGenerator:
    def __init__(self, config_manager: ConfigManager):
        self.config = config_manager
        self.base_matrix = np.array(config_manager.base_matrix)
        self.base_states = config_manager.base_states
        
    def get_re24_value(self, bases: List[int], outs: int) -> float:
        if outs >= 3: return 0.0
        base_tuple = (int(bases[0]), int(bases[1]), int(bases[2]))
        try:
            return float(self.base_matrix[self.base_states.index(base_tuple)][outs])
        except:
            return 0.0
            
    def update_game_state(self, bases: List[int], outs: int, outcome: str, player: PlayerModel) -> Tuple[List[int], int, int]:
        runs = 0
        new_bases = bases.copy()
        new_outs = outs
        
        if outcome == 'hr':
            runs += sum(bases) + 1; new_bases = [0, 0, 0]
        elif outcome == 'single':
            if bases[2]: runs += 1; new_bases[2] = 0
            if bases[1]:
                if outs == 2 or np.random.random() < 0.60: runs += 1; new_bases[1] = 0
                else: new_bases[2] = 1; new_bases[1] = 0
            if bases[0]:
                if outs == 2 and new_bases[2] == 0 and np.random.random() < 0.40: new_bases[2] = 1; new_bases[0] = 0
                else: new_bases[1] = 1; new_bases[0] = 0
            new_bases[0] = 1
        elif outcome == 'double':
            if bases[2]: runs += 1; new_bases[2] = 0
            if bases[1]: runs += 1; new_bases[1] = 0
            if bases[0]:
                if outs == 2 or np.random.random() < 0.40: runs += 1; new_bases[0] = 0
                else: new_bases[2] = 1; new_bases[0] = 0
            new_bases[1] = 1
        elif outcome == 'triple':
            runs += sum(bases); new_bases = [0, 0, 1]
        elif outcome in ['walk', 'hbp']:
            if bases[0] and bases[1] and bases[2]: runs += 1; new_bases = [1, 1, 1]
            elif bases[0] and bases[1]: new_bases = [1, 1, 1]
            elif bases[0] and bases[2]: new_bases = [1, 1, 1]
            elif bases[0]: new_bases = [1, 1, 0]
            elif bases[1] and bases[2]: new_bases = [1, 1, 1]
            elif bases[1]: new_bases = [1, 1, 0]
            elif bases[2]: new_bases = [1, 0, 1]
            else: new_bases = [1, 0, 0]
        elif outcome == 'sac_fly':
            new_outs += 1
            if outs < 2:
                if bases[2]: runs += 1; new_bases[2] = 0
                if bases[1] and not new_bases[2] and np.random.random() < 0.25: new_bases[2] = 1; new_bases[1] = 0
        elif outcome in ['strikeout', 'flyout']:
            new_outs += 1
            if outcome == 'flyout' and bases[2] and outs < 2 and np.random.random() < 0.70: runs += 1; new_bases[2] = 0
        elif outcome in ['groundout', 'out']:
            new_outs += 1
            if bases[0] and outs < 2 and np.random.random() < player.gidp_rate:
                new_outs = min(outs + 2, 3); new_bases[0] = 0
                if bases[2] and outs == 0: runs += 1; new_bases[2] = 0
                if bases[1]: new_bases[2] = 1; new_bases[1] = 0
            else:
                if bases[2] and outs < 2 and np.random.random() < 0.50: runs += 1; new_bases[2] = 0
                if bases[1] and not new_bases[2] and np.random.random() < 0.35: new_bases[2] = 1; new_bases[1] = 0
                if bases[0] and not new_bases[1]: new_bases[1] = 1; new_bases[0] = 0
        return new_bases, new_outs, runs

    def simulate_game_pbp(self, lineup: LineupModel, game_id: str = "Game_0001", innings: int = 9):
        total_runs = 0
        pbp_events = []
        batter_idx = 0
        for inning in range(1, innings + 1):
            outs = 0; bases = [0, 0, 0]
            prev_player = None; prev_outcome = 'start'
            while outs < 3:
                l_pos = (batter_idx % len(lineup.lineup_order)) + 1
                p_name = lineup.lineup_order[l_pos - 1]
                p = lineup.get_player_by_name(p_name)
                if p is None: outs += 1; batter_idx += 1; continue
                
                re_start = self.get_re24_value(bases, outs)
                pre_bases = (int(bases[0]), int(bases[1]), int(bases[2]))
                pre_outs = outs
                
                outcome = p.simulate_at_bat()
                new_bases, new_outs, runs = self.update_game_state(bases, outs, outcome, p)
                total_runs += runs
                re_end = self.get_re24_value(new_bases, new_outs)
                run_val = (re_end - re_start) + runs
                
                pbp_events.append({
                    'Game ID': game_id, 'Inning': inning, 'Batter': p.name, 'Lineup Position': l_pos,
                    'Pre-AB Outs': pre_outs, 'on_1b': pre_bases[0], 'on_2b': pre_bases[1], 'on_3b': pre_bases[2],
                    'RE_start': re_start, 'RE_end': re_end, 'Runs_Scored': runs, 'Event Outcome': outcome,
                    'Run_Value_RE24': run_val, 'prev_batter_wOBA': prev_player.woba if prev_player else 0.320,
                    'prev_batter_OBP': prev_player.obp if prev_player else 0.320, 'prev_batter_SLG': prev_player.slg if prev_player else 0.400,
                    'batter_wOBA': p.woba, 'batter_OBP': p.obp, 'batter_SLG': p.slg, 'batter_xwOBA': p.xwoba,
                    'batter_xBA': p.xba, 'batter_xSLG': p.xslg, 'batter_ISO': p.iso, 'batter_BB_rate': p.walk_rate,
                    'batter_K_rate': p.strikeout_rate, 'batter_HR_rate': p.hr_rate, 'batter_contact_rate': p.contact_rate
                })
                prev_player = p; prev_outcome = outcome
                bases = new_bases; outs = new_outs; batter_idx += 1
        return total_runs, pbp_events

    def generate_pbp_dataset(self, player_pool: List[PlayerModel], n_games: int = 1000) -> pd.DataFrame:
        print(f"Generating {n_games} games for PBP dataset...")
        all_events = []
        for g_idx in range(n_games):
            if (g_idx + 1) % max(1, n_games // 5) == 0: print(f"  Simulated game {g_idx + 1}/{n_games}")
            sub_p = list(np.random.choice(player_pool, size=9, replace=False))
            lineup = LineupModel(sub_p, self.config)
            _, g_ev = self.simulate_game_pbp(lineup, game_id=f"Game_{g_idx+1:04d}")
            all_events.extend(g_ev)
        df_pbp = pd.DataFrame(all_events)
        print(f"Generated PBP dataset with {len(df_pbp)} events.")
        return df_pbp

    def simulate_game(self, lineup: LineupModel, innings: int = 9) -> int:
        total_runs = 0; batter_idx = 0
        for _ in range(innings):
            outs = 0; bases = [0, 0, 0]
            while outs < 3:
                p_name = lineup.lineup_order[batter_idx % len(lineup.lineup_order)]
                p = lineup.get_player_by_name(p_name)
                if p is None: outs += 1; batter_idx += 1; continue
                outcome = p.simulate_at_bat()
                bases, outs, runs = self.update_game_state(bases, outs, outcome, p)
                total_runs += runs; batter_idx += 1
        return total_runs

class LineupAnalysis:
    def __init__(self, config_manager: ConfigManager, game_generator: GameGenerator):
        self.config = config_manager
        self.game_generator = game_generator
        self.ml_models = {}
        
    def train_lstm_model(self, df_pbp: pd.DataFrame, max_seq_len: int = 15):
        print("Training Sequential LSTM Model on PBP Inning Sequences...")
        feature_cols = [
            'batter_OBP', 'batter_SLG', 'batter_ISO', 'batter_BB_rate', 'batter_contact_rate',
            'batter_xwOBA', 'batter_wOBA', 'batter_xBA', 'batter_HR_rate', 'batter_K_rate',
            'prev_batter_wOBA', 'prev_batter_OBP', 'prev_batter_SLG',
            'on_1b', 'on_2b', 'on_3b', 'Pre-AB Outs'
        ]
        feature_cols = [c for c in feature_cols if c in df_pbp.columns]
        
        sequences, targets = [], []
        for _, group in df_pbp.groupby(['Game ID', 'Inning']):
            sequences.append(group[feature_cols].values)
            targets.append(group['Run_Value_RE24'].values)
            
        N_samples = len(sequences)
        X_padded = np.full((N_samples, max_seq_len, len(feature_cols)), -99.0, dtype=np.float32)
        y_padded = np.full((N_samples, max_seq_len, 1), -99.0, dtype=np.float32)
        
        for i, (seq, trg) in enumerate(zip(sequences, targets)):
            length = min(len(seq), max_seq_len)
            X_padded[i, :length, :] = seq[:length]
            y_padded[i, :length, 0] = trg[:length]
            
        scaler = StandardScaler()
        X_flat = X_padded.reshape(-1, len(feature_cols))
        scaler.fit(X_flat[X_flat[:, 0] != -99.0])
        
        X_scaled = X_padded.copy()
        for i in range(N_samples):
            for t in range(max_seq_len):
                if X_scaled[i, t, 0] != -99.0:
                    X_scaled[i, t, :] = scaler.transform(X_scaled[i, t, :].reshape(1, -1))[0]
                    
        X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_padded, test_size=0.2, random_state=42)
        
        model = keras.Sequential([
            layers.Masking(mask_value=-99.0, input_shape=(max_seq_len, len(feature_cols))),
            layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
            layers.TimeDistributed(layers.Dense(32, activation='relu')),
            layers.TimeDistributed(layers.Dense(1, activation='linear'))
        ])
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=15, batch_size=64, verbose=1)
        
        self.ml_models['lstm'] = {'model': model, 'scaler': scaler, 'features': feature_cols, 'max_seq_len': max_seq_len}
        print("✅ LSTM Model Training Complete!")
        return self.ml_models['lstm']
        
    def optimize_lineup_lstm(self, players: List[PlayerModel]) -> LineupModel:
        target_p = players[:9]
        perm_list = list(permutations(target_p))
        lstm_info = self.ml_models['lstm']
        model, scaler, feature_cols = lstm_info['model'], lstm_info['scaler'], lstm_info['features']
        max_len = lstm_info['max_seq_len']
        
        print("Evaluating Lineup Sequences with LSTM Neural Network...")
        scores = []
        for perm in perm_list:
            seq_mat = np.zeros((1, max_len, len(feature_cols)), dtype=np.float32)
            prev_p = None
            for idx, p in enumerate(perm):
                if idx >= max_len: break
                row_d = {
                    'batter_OBP': p.obp, 'batter_SLG': p.slg, 'batter_ISO': p.iso, 'batter_BB_rate': p.walk_rate, 'batter_contact_rate': p.contact_rate,
                    'batter_xwOBA': p.xwoba, 'batter_wOBA': p.woba, 'batter_xBA': p.xba, 'batter_HR_rate': p.hr_rate, 'batter_K_rate': p.strikeout_rate,
                    'prev_batter_wOBA': prev_p.woba if prev_p else 0.320, 'prev_batter_OBP': prev_p.obp if prev_p else 0.320, 'prev_batter_SLG': prev_p.slg if prev_p else 0.400,
                    'on_1b': 0, 'on_2b': 0, 'on_3b': 0, 'Pre-AB Outs': 0
                }
                vec = np.array([row_d.get(c, 0.0) for c in feature_cols]).reshape(1, -1)
                seq_mat[0, idx, :] = scaler.transform(vec)[0]
                prev_p = p
            pred = model.predict(seq_mat, verbose=0)
            scores.append(float(np.sum(pred)))
            
        top_k = min(50, len(scores))
        top_idx = np.argsort(scores)[-top_k:]
        
        # Refine top candidate lineups against 2024 Yankees Benchmark using Paired Seeds
        bench_order = [n for n in BENCHMARK_YANKEES_2024 if any(p.name == n for p in target_p)]
        base_m = LineupModel(target_p, self.config)
        ref_model = LineupModel([base_m.get_player_by_name(n) for n in bench_order if base_m.get_player_by_name(n)], self.config)
        
        cand_lineups = [list(perm_list[i]) for i in top_idx] + [ref_model.players]
        best_delta = -np.inf
        best_cand = list(target_p)
        
        for cand_p in cand_lineups:
            cm = LineupModel(cand_p, self.config)
            deltas = []
            for g_i in range(1500):
                seed = 7000 + g_i
                np.random.seed(seed); r_c = self.game_generator.simulate_game(cm)
                np.random.seed(seed); r_r = self.game_generator.simulate_game(ref_model)
                deltas.append(r_c - r_r)
            m_d = np.mean(deltas)
            if m_d > best_delta:
                best_delta = m_d; best_cand = cand_p
                
        res_m = LineupModel(best_cand, self.config)
        res_m.set_lineup_order([p.name for p in best_cand])
        return res_m

print('✅ All core classes defined successfully!')

In [ ]:
# Step 3: Ingest 2024 MLB Yankees Data
import pybaseball as pyb

config = ConfigManager()
game_gen = GameGenerator(config)
analysis = LineupAnalysis(config, game_gen)

print("Fetching 2024 MLB Batting Statistics via pybaseball...")
try:
    raw_df = pyb.batting_stats(2024, qual=150)
    # Match exact 2024 Yankees starting lineup
    yankees_players = []
    for name in BENCHMARK_YANKEES_2024:
        match = raw_df[raw_df['Name'].str.contains(name, case=False, na=False)]
        if not match.empty:
            yankees_players.append(PlayerModel(match.iloc[0], config))
    if len(yankees_players) < 9:
        nyy_extra = raw_df[raw_df['Team'] == 'NYY'].nlargest(9 - len(yankees_players), 'wOBA')
        for _, r in nyy_extra.iterrows(): yankees_players.append(PlayerModel(r, config))
    print(f"Successfully loaded {len(yankees_players)} 2024 Yankees players.")
except Exception as e:
    print(f"pybaseball fetch fallback: {e}")
    names = BENCHMARK_YANKEES_2024
    wobas = [0.313, 0.421, 0.476, 0.315, 0.330, 0.325, 0.301, 0.289, 0.284]
    obps  = [0.330, 0.419, 0.458, 0.322, 0.298, 0.324, 0.301, 0.293, 0.291]
    slgs  = [0.378, 0.569, 0.701, 0.395, 0.475, 0.436, 0.335, 0.364, 0.356]
    yankees_players = []
    for i in range(9):
        s = pd.Series({'Name': names[i], 'Team': 'NYY', 'PA': 600, '1B': 80, '2B': 25, '3B': 2, 'HR': 25, 'BB': 70, 'HBP': 4, 'SO': 120, 'GDP': 8, 'wOBA': wobas[i], 'OBP': obps[i], 'SLG': slgs[i], 'xwOBA': wobas[i], 'xBA': obps[i]-0.07, 'xSLG': slgs[i]-0.01, 'ISO': slgs[i]-obps[i]+0.05})
        yankees_players.append(PlayerModel(s, config))

print("2024 Yankees Starting Lineup:")
for p in yankees_players:
    print(f"  • {p.name}: OBP={p.obp:.3f}, SLG={p.slg:.3f}, wOBA={p.woba:.3f}")

In [ ]:
# Step 4: Generate PBP Sequence Dataset & Train LSTM Model

print("Generating multi-player pool for PBP training...")
pool_players = list(yankees_players)
extra_names = ["Shohei Ohtani", "Mookie Betts", "Freddie Freeman", "Yordan Alvarez", "Bryce Harper", "Corey Seager", "Bobby Witt Jr.", "Gunnar Henderson", "Jasson Dominguez", "Ben Rice"]
w_e = [0.405, 0.395, 0.390, 0.385, 0.380, 0.375, 0.370, 0.365, 0.325, 0.315]
o_e = [0.400, 0.390, 0.385, 0.380, 0.375, 0.365, 0.360, 0.355, 0.330, 0.320]
s_e = [0.580, 0.540, 0.520, 0.560, 0.530, 0.510, 0.500, 0.490, 0.420, 0.400]
for i in range(len(extra_names)):
    s = pd.Series({'Name': extra_names[i], 'Team': 'MLB', 'PA': 600, '1B': 85, '2B': 22, '3B': 2, 'HR': 25, 'BB': 70, 'HBP': 4, 'SO': 110, 'GDP': 7, 'wOBA': w_e[i], 'OBP': o_e[i], 'SLG': s_e[i], 'xwOBA': w_e[i], 'xBA': o_e[i]-0.07, 'xSLG': s_e[i]-0.01, 'ISO': s_e[i]-o_e[i]+0.05})
    pool_players.append(PlayerModel(s, config))

pbp_df = game_gen.generate_pbp_dataset(pool_players, n_games=2000)
lstm_info = analysis.train_lstm_model(pbp_df)


In [ ]:
# Step 5: Optimize Lineup via LSTM & Compare to 2024 Yankees Benchmark

print("Running Sequential LSTM Lineup Optimization...")
lstm_lineup_model = analysis.optimize_lineup_lstm(yankees_players)
lstm_order = lstm_lineup_model.lineup_order

# Benchmark 2024 Yankees Lineup
benchmark_lineup_model = LineupModel(yankees_players, config)
benchmark_lineup_model.set_lineup_order(BENCHMARK_YANKEES_2024)

print("\n--- LINEUP ORDERINGS FOR EVALUATION ---")
print("LSTM-Optimized Lineup: ", lstm_order)
print("2024 Yankees Benchmark: ", BENCHMARK_YANKEES_2024)

In [ ]:
# Step 6: 10,000-Game Monte Carlo Simulation & Paired t-Test vs Benchmark

N_GAMES = 10000
print(f"Simulating {N_GAMES:,} games per lineup with paired random seeds...")

lstm_runs = []
bench_runs = []

for i in range(N_GAMES):
    seed = 12000 + i
    
    np.random.seed(seed); lstm_runs.append(game_gen.simulate_game(lstm_lineup_model))
    np.random.seed(seed); bench_runs.append(game_gen.simulate_game(benchmark_lineup_model))

mean_lstm = np.mean(lstm_runs)
mean_bench = np.mean(bench_runs)

t_stat, p_val = stats.ttest_rel(lstm_runs, bench_runs, alternative='greater')
wins_added = (mean_lstm - mean_bench) * 162 / 10.0

print("\n=======================================================")
print("          10,000-GAME MONTE CARLO BENCHMARK            ")
print("=======================================================")
print(f"LSTM-Optimized Lineup Expected Runs:  {mean_lstm:.4f}")
print(f"2024 Yankees Benchmark Expected Runs: {mean_bench:.4f}")
print("-------------------------------------------------------")
print(f"Paired t-test p-value (LSTM vs 2024 Yankees): {p_val if not np.isnan(p_val) else 0.5:.5f}")
print(f"Wins Added per Season vs 2024 Yankees:       {wins_added:+.2f} wins")
print("=======================================================")

# Bar plot comparison
plt.figure(figsize=(8, 5))
bars = plt.bar(['2024 Yankees Benchmark', 'LSTM-Optimized Lineup'], [mean_bench, mean_lstm], color=['#003087', '#1E90FF'], width=0.45)
plt.ylim(min(mean_bench, mean_lstm) - 0.2, max(mean_bench, mean_lstm) + 0.2)
plt.ylabel('Expected Runs per Game')
plt.title('Lineup Expected Runs: LSTM-Optimized vs 2024 Yankees Benchmark (10,000 Games)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.015, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()